In [1]:
import pandas as pd
import joblib
from pathlib import Path

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

# Load the cleaned dataset
df = pd.read_csv("../data/employee_attrition_clean.csv")

# Use the same feature set as Days 12–13
features = [
    "TotalMonthsOfExp",
    "TotalOrgsWorked",
    "MonthsInOrg",
    "LastPayIncrementBand",
    "AverageFeedback"
]

X = df[features]
y = df["Attrition"]

# Split the dataset
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Train Random Forest
rf_model = RandomForestClassifier(
    n_estimators=200,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

print("Model training completed.")

Model training completed.


In [3]:
model_path = Path("C:/Users/saqli/Downloads/workspace/Employee-Attrition-Project/models/random_forest_v1.joblib")

joblib.dump(rf_model, model_path)

print("Model saved at:", model_path.resolve())

Model saved at: C:\Users\saqli\Downloads\workspace\Employee-Attrition-Project\models\random_forest_v1.joblib


In [4]:
loaded_model = joblib.load(
    "C:/Users/saqli/Downloads/workspace/Employee-Attrition-Project/models/random_forest_v1.joblib"
)

print("Model loaded successfully.")

Model loaded successfully.


In [5]:
predictions = loaded_model.predict(X_test)

results = pd.DataFrame({
    "Actual": y_test.to_numpy(),
    "Predicted": predictions
})

print(results.head(10))

   Actual  Predicted
0       0          0
1       0          1
2       0          0
3       0          0
4       0          0
5       0          1
6       1          0
7       0          1
8       0          0
9       0          0


In [6]:
original_predictions = rf_model.predict(X_test)
loaded_predictions = loaded_model.predict(X_test)

print(
    "Predictions match:",
    (original_predictions == loaded_predictions).all()
)

Predictions match: True


In [7]:
import pickle

# Save the model
with open("../models/random_forest_v1.pkl", "wb") as file:
    pickle.dump(rf_model, file)

print("Model saved using Pickle.")

Model saved using Pickle.


In [8]:
with open("C:/Users/saqli/Downloads/workspace/Employee-Attrition-Project/models/random_forest_v1.pkl", "rb") as file:
    pickle_model = pickle.load(file)

print("Model loaded using Pickle.")

Model loaded using Pickle.


In [9]:
def predict_employee(employee_data, trained_model):
    """
    Predict the target class for one employee.

    employee_data must contain the same five features
    used during training.
    """

    input_df = pd.DataFrame(
        [employee_data],
        columns=features
    )

    prediction = trained_model.predict(input_df)[0]
    probabilities = trained_model.predict_proba(input_df)[0]

    return {
        "predicted_class": int(prediction),
        "class_0_probability": round(float(probabilities[0]), 3),
        "class_1_probability": round(float(probabilities[1]), 3)
    }

In [10]:
sample_employee = {
    "TotalMonthsOfExp": 60,
    "TotalOrgsWorked": 3,
    "MonthsInOrg": 24,
    "LastPayIncrementBand": 2,
    "AverageFeedback": 3
}

result = predict_employee(sample_employee, loaded_model)

print(result)

{'predicted_class': 1, 'class_0_probability': 0.325, 'class_1_probability': 0.675}


In [11]:
import json
from datetime import datetime, timezone

metadata = {
    "model_name": "Employee Attrition Random Forest",
    "version": "1.0.0",
    "algorithm": "RandomForestClassifier",
    "features": features,
    "target": "Attrition",
    "random_state": 42,
    "trained_at_utc": datetime.now(timezone.utc).isoformat(),
    "model_file": "random_forest_v1.joblib"
}

with open("../models/model_metadata.json", "w") as file:
    json.dump(metadata, file, indent=4)

print("Model metadata saved.")

Model metadata saved.
